In [ ]:
"""
train.py - Explainable Diabetes Prediction | 2026 Edition
==========================================================
Dataset : PIMA Indians Diabetes (768 patients, 8 clinical features)
Model   : XGBoost (XGBClassifier) only
Pipeline: median imputation -> SMOTE -> XGBoost, tuned with GridSearchCV (ROC-AUC)
XAI     : SHAP TreeExplainer (global summary plot + waterfall for highest-risk patient)

Outputs (all written to ./assets/):
    model.pkl            best fitted pipeline (imputer + XGBoost; SMOTE is train-time only)
    confusion_matrix.png
    roc_curve.png
    shap_summary.png
    shap_waterfall.png
    metrics.json         CV ROC-AUC, Test ROC-AUC and other metrics
    results_table.csv    metrics as a table (paste into README / paper)

Run in Google Colab:  upload this file, then  `!python train.py`
(or paste the whole file into one cell).
"""
import importlib.util
import json
import subprocess
import sys
import warnings
from pathlib import Path

# --------------------------------------------------------------------------- #
# Colab convenience: install anything that is missing (Colab already ships with
# xgboost / scikit-learn / pandas / matplotlib, but NOT imbalanced-learn / shap).
# --------------------------------------------------------------------------- #
for _import_name, _pip_name in [
    ("imblearn", "imbalanced-learn"),
    ("shap", "shap"),
    ("xgboost", "xgboost"),
]:
    if importlib.util.find_spec(_import_name) is None:
        print(f"Installing {_pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pip_name])

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from imblearn.over_sampling import SMOTE                 # 2026 Edition: SMOTE for imbalance
from imblearn.pipeline import Pipeline as ImbPipeline    # resampling inside CV = no leakage
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# --------------------------------------------------------------------------- #
# Configuration
# --------------------------------------------------------------------------- #
RANDOM_STATE = 42
DATA_URL = (
    "https://raw.githubusercontent.com/jbrownlee/Datasets/master/"
    "pima-indians-diabetes.data.csv"
)
COLUMNS = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome",
]
FEATURES = COLUMNS[:-1]
TARGET = "Outcome"
# In PIMA a 0 in these columns is a missing measurement (a living patient cannot
# have 0 glucose or 0 BMI), so we treat them as NaN. Pregnancies/Age zeros are valid.
ZERO_AS_NAN = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

# assets/ folder is created if it does not exist
ASSETS = Path("assets")
ASSETS.mkdir(parents=True, exist_ok=True)


def main() -> None:
    # ----------------------------------------------------------------------- #
    # 1) Load data and mark impossible zeros as missing
    # ----------------------------------------------------------------------- #
    df = pd.read_csv(DATA_URL, header=None, names=COLUMNS)
    df[ZERO_AS_NAN] = df[ZERO_AS_NAN].replace(0, np.nan)
    print(f"Loaded {len(df)} patients | diabetes prevalence = {df[TARGET].mean():.1%}")
    print("Missing values after zero->NaN:\n", df[ZERO_AS_NAN].isna().sum().to_string(), "\n")

    X, y = df[FEATURES], df[TARGET]

    # ----------------------------------------------------------------------- #
    # 2) Stratified 80/20 split. Done BEFORE imputation and SMOTE so that the
    #    test set never influences medians or synthetic samples (no leakage).
    # ----------------------------------------------------------------------- #
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
    )

    # Illustration only: show what SMOTE does to the class balance of the train set.
    _imp_demo = SimpleImputer(strategy="median").fit_transform(X_train)
    _, y_demo = SMOTE(random_state=RANDOM_STATE).fit_resample(_imp_demo, y_train)
    print("Train class counts BEFORE SMOTE:", y_train.value_counts().to_dict())
    print("Train class counts AFTER  SMOTE:", pd.Series(y_demo).value_counts().to_dict(), "\n")

    # ----------------------------------------------------------------------- #
    # 3) Pipeline: median imputation -> SMOTE -> XGBoost
    #    2026 Edition: SMOTE for imbalance, applied only to the training part of
    #    each CV fold (imblearn pipelines skip samplers at predict time).
    # ----------------------------------------------------------------------- #
    pipeline = ImbPipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("smote", SMOTE(random_state=RANDOM_STATE)),
            ("xgb", XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=RANDOM_STATE,
                n_jobs=1,  # parallelism is handled by GridSearchCV below
            )),
        ]
    )

    # GridSearchCV over n_estimators, max_depth, learning_rate (scoring = ROC-AUC)
    param_grid = {
        "xgb__n_estimators": [100, 200, 300],
        "xgb__max_depth": [3, 4, 5],
        "xgb__learning_rate": [0.01, 0.05, 0.1],
    }
    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=-1,
        refit=True,
        verbose=1,
    )
    grid.fit(X_train, y_train)

    best_pipe = grid.best_estimator_
    best_params = {k.replace("xgb__", ""): v for k, v in grid.best_params_.items()}
    cv_auc = float(grid.best_score_)
    print(f"\nBest params : {best_params}")
    print(f"CV ROC-AUC  : {cv_auc:.4f}")

    # ----------------------------------------------------------------------- #
    # 4) Evaluate once on the untouched test set
    # ----------------------------------------------------------------------- #
    proba = best_pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    test_auc = float(roc_auc_score(y_test, proba))
    print(f"Test ROC-AUC: {test_auc:.4f}\n")
    print(classification_report(y_test, pred, target_names=["No diabetes", "Diabetes"]))

    metrics = {
        "cv_roc_auc": round(cv_auc, 4),
        "test_roc_auc": round(test_auc, 4),
        "test_pr_auc": round(float(average_precision_score(y_test, proba)), 4),
        "test_accuracy": round(float(accuracy_score(y_test, pred)), 4),
        "test_precision": round(float(precision_score(y_test, pred)), 4),
        "test_recall": round(float(recall_score(y_test, pred)), 4),
        "test_f1": round(float(f1_score(y_test, pred)), 4),
        "best_params": best_params,
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
        "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
    }
    with open(ASSETS / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    # results_table.csv - flat table of the headline numbers
    pd.DataFrame(
        {
            "Metric": ["CV ROC-AUC", "Test ROC-AUC", "Test PR-AUC", "Accuracy",
                       "Precision", "Recall", "F1"],
            "Value": [metrics["cv_roc_auc"], metrics["test_roc_auc"], metrics["test_pr_auc"],
                      metrics["test_accuracy"], metrics["test_precision"],
                      metrics["test_recall"], metrics["test_f1"]],
        }
    ).to_csv(ASSETS / "results_table.csv", index=False)

    # ----------------------------------------------------------------------- #
    # 5) Plots: confusion matrix + ROC curve
    # ----------------------------------------------------------------------- #
    fig, ax = plt.subplots(figsize=(5, 4.5))
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, display_labels=["No diabetes", "Diabetes"],
        cmap="Blues", colorbar=False, ax=ax,
    )
    ax.set_title("Confusion matrix (held-out test set)")
    fig.tight_layout()
    fig.savefig(ASSETS / "confusion_matrix.png", dpi=200)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(5, 4.5))
    RocCurveDisplay.from_predictions(
        y_test, proba, name=f"XGBoost + SMOTE (AUC = {test_auc:.3f})", ax=ax
    )
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Chance")
    ax.set_title("ROC curve (held-out test set)")
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(ASSETS / "roc_curve.png", dpi=200)
    plt.close(fig)

    # ----------------------------------------------------------------------- #
    # 6) Explainable AI: SHAP with TreeExplainer
    #    SHAP must see the same (imputed) numbers the XGBoost model was trained on,
    #    so we push the test set through the fitted imputer first.
    # ----------------------------------------------------------------------- #
    imputer = best_pipe.named_steps["imputer"]
    xgb_model = best_pipe.named_steps["xgb"]
    X_test_imp = imputer.transform(X_test)  # numpy array, same column order as FEATURES

    explainer = shap.TreeExplainer(xgb_model)
    sv = explainer(X_test_imp)  # SHAP values in log-odds space
    # Rebuild the Explanation with real feature names so the plots are readable.
    explanation = shap.Explanation(
        values=sv.values,
        base_values=sv.base_values,
        data=X_test_imp,
        feature_names=FEATURES,
    )

    # Global explanation: which features drive risk across all test patients?
    plt.figure()
    shap.summary_plot(explanation, X_test_imp, feature_names=FEATURES, show=False)
    plt.title("SHAP summary - global feature impact")
    plt.tight_layout()
    plt.savefig(ASSETS / "shap_summary.png", dpi=200, bbox_inches="tight")
    plt.close()

    # Local explanation: WHY is the highest-risk test patient high risk?
    top_idx = int(np.argmax(proba))
    plt.figure()
    shap.plots.waterfall(explanation[top_idx], max_display=len(FEATURES), show=False)
    plt.title(f"SHAP waterfall - highest-risk test patient (predicted risk {proba[top_idx]:.0%})")
    plt.tight_layout()
    plt.savefig(ASSETS / "shap_waterfall.png", dpi=200, bbox_inches="tight")
    plt.close()

    # ----------------------------------------------------------------------- #
    # 7) Save the best model.
    #    model.pkl is the full fitted pipeline: raw patient rows (NaN allowed) ->
    #    median imputation -> XGBoost. Use: joblib.load("assets/model.pkl").predict_proba(df)
    # ----------------------------------------------------------------------- #
    joblib.dump(best_pipe, ASSETS / "model.pkl")

    print("\nDone. Files in assets/:", sorted(p.name for p in ASSETS.iterdir()))


if __name__ == "__main__":
    main()